In [4]:
# Install the Hugging Face Hub client in the active notebook runtime.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "huggingface_hub",
])

0

In [5]:
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR = PROJECT_DIR / "multisite-ppg"

print(f"Running in Colab: {IN_COLAB}")
print(f"Dataset folder: {DATA_DIR}")

Running in Colab: True
Dataset folder: /content/multisite-ppg


In [6]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="anonymous-ppg-dataset/multisite-ppg-submission",
    repo_type="dataset",
    allow_patterns="sample_data/*",
    local_dir=str(DATA_DIR),
)

sample_files = sorted((DATA_DIR / "sample_data").glob("**/*"))
print(f"Downloaded {sum(path.is_file() for path in sample_files)} files")

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Downloaded 23 files


In [10]:
import numpy as np

# `sample_data` is a directory, not a single `sample_data.npz` archive.
# Load one of the actual pre-windowed PPG sample files downloaded above.
sample_npz_path = (
    DATA_DIR
    / "sample_data"
    / "ppg_windowed_data"
    / "P7"
    / "alignment_windows_P7_Earring.npz"
)

if not sample_npz_path.is_file():
    raise FileNotFoundError(
        f"{sample_npz_path} was not found. Run the download cell above first."
    )

data = np.load(sample_npz_path)

print(f"Loaded: {sample_npz_path.name}")
print("Available arrays:", data.files)
print("PPG shape:", data["ppg_green"].shape)

Loaded: alignment_windows_P7_Earring.npz
Available arrays: ['t0_ms', 't1_ms', 'ecg', 'ecg_valid_len', 'ppg_fs', 'ppg_ir', 'ppg_green', 'hr_gt', 'n_peaks', 'accel_x', 'accel_y', 'accel_z']
PPG shape: (25668, 800)


In [11]:
from pathlib import Path

sample_dir = DATA_DIR / "sample_data"
print(sample_dir.exists())
print("file count:", sum(p.is_file() for p in sample_dir.glob("**/*")))
print("total size GB:", sum(p.stat().st_size for p in sample_dir.glob("**/*") if p.is_file()) / 1e9)

True
file count: 23
total size GB: 1.418792957


In [15]:
import matplotlib.pyplot as plt 
from  scipy.signal import butter , filtfilt,detrend 

In [16]:
def basic_ppg_preprocess(signal,fs=100,lowcut=0.5,highcut=8.0,order=4):
    signal=np.asarray(signal) 
    signal=np.squeeze(signal) 
    signal=np.nan_to_num(signal,nan=0.0,posinf=0.0,neginf=0.0)
    signal=detrend(signal)
    nyquist=fs/2 
    low = lowcut/nyquist 
    high = highcut/nyquist 
    b,a = butter(order,[low,high],btype='band') 
    filtered_signal = filtfilt(b,a,signal) 
    mean = np.mean(filtered_signal) 
    std = np.std(filtered_signal) 
    if std > 1e-8 : 
        normalized_signal = (filtered_signal - mean)/std 
    else : 
        normalized_signal = filtered_signal - mean 
    return normalized_signal 


In [18]:
raw_ppg=data['ppg_green'] 
print("Raw PPG shape:", raw_ppg.shape) 
processed_ppg =basic_ppg_preprocess(raw_ppg,fs=100,lowcut=0.5,highcut=8.0,order=4)

Raw PPG shape: (25668, 800)


In [ ]:
time = np.arange(len(raw_ppg))/100.0 
plt.figure(figsize=(12,6)) 
plt.subplot(2,1,1) 
plt.plot(time,raw_ppg) 
plt.xlabel('Time (s)')
plt.ylabel('Raw PPG')
plt.show()

plt.subplot(2,1,2) 
plt.plot(time,processed_ppg) 
plt.xlabel('Time (s)')
plt.ylabel('Processed PPG')
plt.show()